# Task 6 — Idempotent replay verification

```{admonition} Evidence summary
:class: note

| Field | Value |
|---|---|
| Evidence source | `screenshots/replay/stage3_replay_manifest.json` and its hashed artifacts |
| Command | `bash scripts/run_stage3_evidence.sh`; `bash scripts/finalize_stage3_evidence.sh` |
| Replay target | `src/datasets/__init__.py` only |
| Captured at | `2026-07-23T06:56:48Z` |
```

## Approach and reasoning

The workflow restarts Spark with the baseline checkpoint, changes one pinned
source file, replays only that file with a new `run_id`, waits for both
consumers, and then removes stale graph entities. MongoDB replaces the target
document while unchanged documents remain byte-for-byte equivalent.

## Key implementation excerpts

```{literalinclude} ../scripts/run_stage3_evidence.sh
:language: bash
:lines: 137-200
:caption: Deterministic mutation, single-file replay, consumer wait, and cleanup
```

```{literalinclude} ../neo4j/cleanup_stale.cypher
:language: text
:lines: 1-11
:caption: File-scoped deletion of graph entities from older runs
```

## Executed evidence


In [1]:
from pathlib import Path
import json

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "screenshots/replay/stage3_replay_manifest.json").exists()
)
MANIFEST_PATH = ROOT / "screenshots/replay/stage3_replay_manifest.json"
manifest = json.loads(MANIFEST_PATH.read_text())
assert manifest["stage"] == 3 and manifest["status"] == "pass"
print("manifest:", MANIFEST_PATH.relative_to(ROOT))
print("captured_at:", manifest["run"]["captured_at"])


manifest: screenshots/replay/stage3_replay_manifest.json
captured_at: 2026-07-23T06:56:48Z


## Modified source proof

The captured unified diff makes the single modified file and exact change
reviewable without requiring access to the live clone.


In [2]:
patch = (ROOT / "screenshots/replay/source_patch.diff").read_text().strip()
print("source patch:")
print(patch)
assert patch.count("--- a/") == patch.count("+++ b/") == 1
assert "src/datasets/__init__.py" in patch
assert "LAB04_REPLAY_MARKER" in patch


source patch:
--- a/src/datasets/__init__.py
+++ b/src/datasets/__init__.py
@@ -16 +16,2 @@
-__version__ = "5.0.1.dev0"
+__version__: str = "5.0.1.dev0+lab04-replay"
+LAB04_REPLAY_MARKER = "replay_v2"


## Modified file and run identity


In [3]:
source = manifest["source"]
print("target:", source["target_file"])
print("file_id:", source["file_id"])
print("content_hash:", source["before_content_hash"], "->", source["after_content_hash"])
print("run_id:", manifest["run"]["baseline_run_id"], "->", manifest["run"]["replay_run_id"])
assert source["before_content_hash"] != source["after_content_hash"]
assert manifest["run"]["baseline_run_id"] != manifest["run"]["replay_run_id"]
assert source["source_restored"] is True


target: src/datasets/__init__.py
file_id: 6c39568a6a11c430
content_hash: 74ab176247c9ba61e09843280c18cf3a9ac690403a5d35411a75d097db721939 -> 6db67191532027fbf91190815efa70380d79c2eda9e61136a41a1d1a8a8e0735
run_id: 8c3efe458b3b43eaa1399d7740848d3f -> stage3-replay-20260723T065648Z


## Kafka and Spark checkpoint proof


In [4]:
print("Kafka replay delta:", manifest["kafka"]["delta"])
print(
    "Spark metadata offsets:",
    manifest["spark"]["metadata_offset_before"], "->",
    manifest["spark"]["metadata_offset_after_restart"], "->",
    manifest["spark"]["metadata_offset_after_replay"],
)
assert manifest["kafka"]["delta"]["cpg.nodes"] > 0
assert manifest["kafka"]["delta"]["cpg.edges"] >= 0
assert manifest["kafka"]["delta"]["cpg.metadata"] == 1
assert manifest["kafka"]["delta"]["cpg.errors"] == 0
before = manifest["spark"]["metadata_offset_before"]
assert manifest["spark"]["metadata_offset_after_restart"] == before
assert manifest["spark"]["metadata_offset_after_replay"] == before + 1


Kafka replay delta: {'cpg.edges': 16, 'cpg.errors': 0, 'cpg.metadata': 1, 'cpg.nodes': 61}
Spark metadata offsets: 138 -> 138 -> 139


## Neo4j cleanup and MongoDB replacement


In [5]:
neo4j = manifest["neo4j"]
mongo = manifest["mongodb"]
print(
    "Neo4j explicit nodes:",
    neo4j["before"]["explicit_nodes"], "->",
    neo4j["pre_cleanup"]["explicit_nodes"], "->",
    neo4j["after"]["explicit_nodes"],
)
print(
    "Neo4j relationships:",
    neo4j["before"]["edges"], "->",
    neo4j["pre_cleanup"]["edges"], "->",
    neo4j["after"]["edges"],
)
print("stale deleted:", neo4j["stale_deleted"])
print(
    "MongoDB documents:",
    mongo["documents_before"], "->", mongo["documents_after_replay"],
    "; unchanged:", mongo["unchanged_documents"],
)
assert neo4j["stale_deleted"] == {
    "nodes": neo4j["pre_cleanup"]["stale_nodes"],
    "edges": neo4j["pre_cleanup"]["stale_edges"],
}
assert neo4j["after"]["duplicate_node_groups"] == 0
assert neo4j["after"]["duplicate_edge_groups"] == 0
assert mongo["documents_after_replay"] == mongo["documents_before"] == 138
assert mongo["unchanged_documents"] == 137


Neo4j explicit nodes: 133263 -> 133288 -> 133267
Neo4j relationships: 38141 -> 38144 -> 38142
stale deleted: {'edges': 2, 'nodes': 21}
MongoDB documents: 138 -> 138 ; unchanged: 137


## What this chapter proves

| Requirement | Evidence |
|---|---|
| One modified file | The executed cell prints the real unified source patch. |
| Checkpoint skips unchanged offsets | Restart stays at 138; the one replay event advances it to 139. |
| Neo4j convergence | Stale entities are deleted and duplicate groups remain zero. |
| MongoDB convergence | Document count stays 138 and 137 unchanged files retain their documents. |

## Database UI evidence

```{admonition} Database UI evidence
:class: important

| Store | UI artifact | Meaning |
|---|---|---|
| Neo4j | `neo4j_after_cleanup.png` | Target graph after stale cleanup |
| MongoDB | `mongodb_after_replay.png` | Target document after replacement |
```

![Neo4j replay state](../screenshots/replay/neo4j_after_cleanup.png)

![MongoDB replay document](../screenshots/replay/mongodb_after_replay.png)

## Reflection

```{admonition} Closing reflection
:class: tip

| Point | Result |
|---|---|
| Worked | Single-file replay, persisted checkpoint, graph cleanup, and Mongo replacement form one auditable chain. |
| Failed | The old partial baseline treated expected Kafka append-log replay events as invalid duplicates. |
| Resolution | The full baseline separates event repetition from duplicate database state and waits for consumer lag zero. |
| Limitation | Structural AST IDs can change after edits, so replay still needs file-scoped stale cleanup. |
```
